#| export
# pyskills.edit
> Text editing tools for modifying files and ipynb notebook cells. Each operation — insert, replace, delete lines, and string(s) replacement — has both a `file_*` and `cell_*` variant. All return unified diffs showing what changed, or `"none: No changes."`, or `"error: ..."`.
> 
> ## File editing
> 
> File tools take a filesystem path as the first argument, e.g:
> 
>     await file_str_replace('myfile.py', 'old_name', 'new_name')
>     await file_del_lines('myfile.py', 2, 4)
> 
> ## Ipynb file cell editing
> 
> Cell tools take an ipynb path and a cell id, e.g:
> 
>     await cell_replace_lines('nb.ipynb', cell_id, 2, 3, 'replaced')
>     await cell_insert_line('nb.ipynb', cell_id, 0, 'first line')
> 
> Use `view_nb` to view the whole notebook. Use `view_cell` to see a cell's source with line numbers before editing.
> 
> ## Line filtering
> 
> `*_str_replace`, `*_strs_replace`, and `*_del_lines` support `re_filter` and `invert_filter` for targeting only lines matching (or not matching) a regex, like ex's `g//` and `g!//`. Combined with `start_line`/`end_line` to restrict to a region, e.g:
> 
>     await file_del_lines('myfile.py', 1, -1, re_filter=r'^\s*#')       # delete all comment lines

In [ ]:
#| default_exp edit

In [ ]:
#| export
import difflib,re
from pathlib import Path
from tempfile import TemporaryDirectory

from fastcore.utils import maybe_await,llmtool
from fastcore.meta import splice_sig

In [ ]:
from fastcore.test import test_eq

In [ ]:
#| export
_file_edit_doc = """
This is a *file* editing function.

File editing standard parameter is `path`: Path to the file to modify

returns: diff of changes, or "none: No changes.", or "error: ..."
"""

In [ ]:
#TODO @llmtool

In [ ]:
#| export
def file_edit(f, name=None):
    async def wrapper(path:str, *args, **kw):
        path = Path(path).expanduser()
        text = path.read_text()
        try: new_text = await maybe_await(f(text, *args, **kw))
        except ValueError as e: return f'error: {e}'
        Path(path).write_text(new_text)
        diff = '\n'.join(list(difflib.unified_diff(text.splitlines(), new_text.splitlines(), n=1, lineterm=''))[2:])
        return diff or 'none: No changes.'
    res = splice_sig(wrapper, f, 'text')
    if name: res.__name__ = res.__qualname__ = name
    res.__doc__ = (f.__doc__ or '') + _file_edit_doc
    return llm_tool(res)

In [ ]:
_tmp = TemporaryDirectory()
_test_path = f'{_tmp.name}/test.txt'
_test_content = 'alpha\nbeta\ngamma\ndelta\n'
def _test_txt(): return Path(_test_path).read_text()
Path(_test_path).write_text(_test_content)
_test_path

'/var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/tmptbu9drz1/test.txt'

In [ ]:
#| export
def insert_line(
    text:str,
    insert_line:int, # The 1-based line number after which to insert the text (0: before 1st line, n>=1: after nth line)
    new_str:str, # The text to insert
):
    "Insert new_str at specified line number"
    lines = text.splitlines()
    if not (0 <= insert_line <= len(lines)): raise ValueError(f'Invalid line {insert_line}. Valid range: 0-{len(lines)}')
    lines.insert(insert_line, new_str)
    return '\n'.join(lines)

file_insert_line = file_edit(insert_line, 'file_insert_line')

In [ ]:
res = await file_insert_line(_test_path, 0, 'first')
test_eq(_test_txt().splitlines()[0], 'first')
print(res)

@@ -1 +1,2 @@
+first
 alpha


In [ ]:
#| export
def str_replace(
    text:str,
    old_str:str, # Text to find and replace
    new_str:str, # Text to replace with
    start_line:int=None, # Optional 1-based start line to limit search
    end_line:int=None, # Optional 1-based end line to limit search
    n_matches:int=None, # Max replacements (None=all)
    re_filter:str=None, # If provided, only process lines matching this regex (like g// in ex)
    invert_filter:bool=False, # Invert the filter (like g!// in ex)
):
    "Replace occurrence(s) of old_str with new_str"
    def _repl(s, label=''):
        if s.count(old_str) == 0: raise ValueError(f"Text not found{label}: {repr(old_str)}")
        return s.replace(old_str, new_str) if n_matches is None else s.replace(old_str, new_str, n_matches)
    if re_filter or start_line or end_line:
        lines = text.splitlines(True)
        s,e = (start_line or 1)-1, end_line or len(lines)
        if not re_filter: return ''.join(lines[:s]) + _repl(''.join(lines[s:e]), f' in lines {s+1}-{e}') + ''.join(lines[e:])
        pat = re.compile(re_filter)
        matched = [i for i in range(s,e) if bool(pat.search(lines[i])) != invert_filter]
        if not matched: return text
        for i in matched: lines[i] = _repl(lines[i], f' on line {i+1}')
        return ''.join(lines)
    return _repl(text)

file_str_replace = file_edit(str_replace, 'file_str_replace')

In [ ]:
res = await file_str_replace(_test_path, 'beta', 'BETA')
assert 'BETA' in _test_txt(), f"Expected 'BETA' in file"
print(res)

@@ -2,3 +2,3 @@
 alpha
-beta
+BETA
 gamma


In [ ]:
#| export
def strs_replace(
    text:str,
    old_strs:list[str], # List of strings to find and replace
    new_strs:list[str], # List of replacement strings (must match length of old_strs)
    start_line:int=None, # Optional 1-based start line to limit search
    end_line:int=None, # Optional 1-based end line to limit search
    n_matches:int=None, # Max replacements per string (None=all)
    re_filter:str=None, # If provided, only process lines matching this regex (like g// in ex)
    invert_filter:bool=False, # Invert the filter (like g!// in ex)
):
    "Replace multiple strings simultaneously"
    if not isinstance(old_strs, list): raise ValueError(f"`old_strs` should be a list[str] but got {type(old_strs)}")
    if not isinstance(new_strs, list): raise ValueError(f"`new_strs` should be a list[str] but got {type(new_strs)}")
    if len(old_strs) != len(new_strs): raise ValueError(f"Length mismatch: {len(old_strs)} old_strs vs {len(new_strs)} new_strs")
    for idx,(old_str,new_str) in enumerate(zip(old_strs, new_strs)):
        text = str_replace(text, old_str, new_str, start_line=start_line, end_line=end_line,
                            n_matches=n_matches, re_filter=re_filter, invert_filter=invert_filter)
    return text

file_strs_replace = file_edit(strs_replace, 'file_strs_replace')

In [ ]:
res = await file_strs_replace(_test_path, ['gamma', 'delta'], ['GAMMA', 'DELTA'])
test_eq(_test_txt().splitlines()[-2:], ['GAMMA', 'DELTA'])
print(res)

@@ -3,3 +3,3 @@
 BETA
-gamma
-delta
+GAMMA
+DELTA


In [ ]:
#| export
def _norm_lines(n:int, start:int, end:int=None):
    "Normalize and validate line range. Returns (start, end) or raises ValueError."
    if end is None: end = start
    if end < 0: end = n + end + 1
    if not (1 <= start <= n): raise ValueError(f'Invalid start line {start}. Valid range: 1-{n}')
    if not (start <= end <= n): raise ValueError(f'Invalid end line {end}. Valid range: {start}-{n}')
    return start, end

In [ ]:
#| export
def replace_lines(
    text:str,
    start_line:int, # Starting line number to replace (1-based indexing)
    end_line:int=None, # Ending line number to replace (1-based, inclusive, negative counts from end, None for single line)
    new_content:str='', # New content to replace the specified lines
):
    "Replace line range with new content"
    lines = text.splitlines(keepends=True)
    s,e = _norm_lines(len(lines), start_line, end_line)
    if lines and new_content and not new_content.endswith('\n'): new_content += '\n'
    lines[s-1:e] = [new_content] if new_content else []
    return ''.join(lines)

file_replace_lines = file_edit(replace_lines, 'file_replace_lines')

In [ ]:
res = await file_replace_lines(_test_path, 2, 3, 'two\nthree\n')
test_eq(_test_txt().splitlines()[1:3], ['two', 'three'])
print(res)

@@ -1,4 +1,4 @@
 first
-alpha
-BETA
+two
+three
 GAMMA


In [ ]:
#| export
def del_lines(
    text:str,
    start_line:int, # Starting line number to delete (1-based indexing)
    end_line:int=None, # Ending line number to delete (1-based, inclusive, negative counts from end, None for single line)
    re_filter:str=None, # If provided, only delete lines matching this regex (like g// in ex)
    invert_filter:bool=False, # Invert the filter (like g!// in ex)
):
    "Delete line range"
    lines = text.splitlines(keepends=True)
    s,e = _norm_lines(len(lines), start_line, end_line)
    if re_filter:
        pat = re.compile(re_filter)
        matched = {i for i in range(s-1,e) if bool(pat.search(lines[i])) != invert_filter}
        if not matched: return text
        lines = [l for i,l in enumerate(lines) if i not in matched]
    else: del lines[s-1:e]
    return ''.join(lines)

file_del_lines = file_edit(del_lines, 'file_del_lines')

In [ ]:
res = await file_del_lines(_test_path, 1)
test_eq(_test_txt().splitlines()[0], 'two')
print(res)

@@ -1,2 +1 @@
-first
 two


In [ ]:
#| export
_cell_edit_doc = f"""
Be sure you've called `view_cell(…)` to ensure you know the line nums.

Cell editing standard parameters:

id: Cell id to edit, or list of ids, or 'all' for all messages in file
fname: ipynb to get info for
update_output: If True, replace in output instead of content

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages"""

In [ ]:
#| export
def _cell_edit(f, name=None):
    async def wrapper(fname:str, id:str|list[str], *args, update_output:bool=False, **kw):
        nb = Notebook.open(fname)
        def _one(cid):
            cell = nb[cid]
            text = str(cell.outputs) if update_output else cell.source
            if not text: return f"error: Cell has no {'output' if update_output else 'source'}"
            try: new_text = f(text, *args, **kw)
            except ValueError as e: return f'error: {e}'
            if update_output: cell.outputs = ast.literal_eval(new_text)
            else: nb[cid] = new_text
            diff = '\n'.join(list(difflib.unified_diff(text.splitlines(), new_text.splitlines(), n=1, lineterm=''))[2:])
            return diff or 'none: No changes.'
        if isinstance(id, list) or id == 'all':
            if id == 'all': id = [c.id for c in nb.cells]
            res = [(cid, r) for cid in id if not (r := _one(cid)).startswith(('error:', 'none:'))]
        else: res = _one(id)
        nb.save()
        return res
    res = splice_sig(wrapper, f, 'text')
    if name: res.__name__ = res.__qualname__ = name
    res.__doc__ = (f.__doc__ or '') + _cell_edit_doc
    return res

In [ ]:
#| export
from fastcore.nbio import *

In [ ]:
_nb_path = f'{_tmp.name}/test.ipynb'
_tnb = Notebook(new_nb([_test_content, 'other cell']))
_tnb.save(_nb_path)
_cid, _oid = _tnb[0].id, _tnb[1].id
def _nb_src(): return Notebook.open(_nb_path)[_cid].source
_cid, _oid

('819c0118', '9446f307')

In [ ]:
#| export
cell_insert_line = _cell_edit(insert_line, 'cell_insert_line')
cell_str_replace = _cell_edit(str_replace, 'cell_str_replace')
cell_strs_replace = _cell_edit(strs_replace, 'cell_strs_replace')
cell_replace_lines = _cell_edit(replace_lines, 'cell_replace_lines')
cell_del_lines = _cell_edit(del_lines, 'cell_del_lines')

In [ ]:
res = await cell_insert_line(_nb_path, _cid, 0, 'first')
test_eq(_nb_src().splitlines()[0], 'first')
print(res)

@@ -1 +1,2 @@
+first
 alpha


In [ ]:
res = await cell_str_replace(_nb_path, _cid, 'beta', 'BETA')
assert 'BETA' in _nb_src(), f"Expected 'BETA' in cell"
print(res)

@@ -2,3 +2,3 @@
 alpha
-beta
+BETA
 gamma


In [ ]:
res = await cell_strs_replace(_nb_path, _cid, ['gamma', 'delta'], ['GAMMA', 'DELTA'])
test_eq(_nb_src().splitlines()[-2:], ['GAMMA', 'DELTA'])
print(res)

@@ -3,3 +3,3 @@
 BETA
-gamma
-delta
+GAMMA
+DELTA


In [ ]:
res = await cell_replace_lines(_nb_path, _cid, 2, 3, 'two\nthree\n')
test_eq(_nb_src().splitlines()[1:3], ['two', 'three'])
print(res)

@@ -1,4 +1,4 @@
 first
-alpha
-BETA
+two
+three
 GAMMA


In [ ]:
res = await cell_del_lines(_nb_path, _cid, 1)
test_eq(_nb_src().splitlines()[0], 'two')
print(res)

@@ -1,2 +1 @@
-first
 two


In [ ]:
#| export
def view_cell(fname:str, id:str, nums:bool=True):
    "Show cell source with optional line numbers"
    return Notebook.open(fname).view(id, nums=nums)

`view_cell` displays a cell's source with line numbers, useful for checking content before editing:

In [ ]:
print(view_cell(_nb_path, _cid))

     1 │ two
     2 │ three
     3 │ GAMMA
     4 │ DELTA


In [ ]:
#| export
def view_nb(fname:str, incl_out:bool=False):
    "Show notebook source as concise xml, optionally including output if `incl_out`"
    nb = Notebook.open(fname)
    return repr(nb) if incl_out else nb.concise

In [ ]:
view_nb(_nb_path)

'<nb path="test.ipynb"><code id="819c0118">two\nthree\nGAMMA\nDELTA</code><code id="9446f307">other cell</code></nb>'

In [ ]:
#| export
from pyskills.core import PosAllowPolicy,AllowPolicy,allow

In [ ]:
#| export
_wp = PosAllowPolicy(0)
allow({file_insert_line: _wp, file_str_replace: _wp, file_strs_replace: _wp, file_replace_lines: _wp, file_del_lines: _wp,
       cell_insert_line: _wp, cell_str_replace: _wp, cell_strs_replace: _wp, cell_replace_lines: _wp, cell_del_lines: _wp})
allow(view_cell, view_nb)

In [ ]:
#| export
class _NbWritePolicy(AllowPolicy):
    "Check Notebook.path and optional path arg"
    def __call__(self, obj, args, kwargs, ok_dests):
        p = kwargs.get('path', args[0] if args else None) or obj.path
        if p is not None: chk_dest(p, ok_dests)

allow({Notebook: ['open', 'find', 'view', 'add', 'md', 'move']})
allow({Notebook: [('save', _NbWritePolicy())]})

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()